# Phase 2: classify each account with Decision-1

Every account in the crawl store with a usable website goes to Decision-1 in one call: the most useful pages of its
website and our questions (`questions.py`). The rules in `decide.py` turn the answer into an industry, and the report
explains each one. How an account is decided is on the report's first two sheets and in the README.

The same company entered more than once (same website, same name and DBA) gets one call, and every record of it gets
that answer (the answers file's `same_as` column says whose).

Safe to stop and re-run at any time: answered accounts are skipped. The API key is read from the `.env` file.

## Settings

In [ ]:
import classify, crawl, datasets, report, results

SET = "crm_pilot"  # which accounts: "zoominfo_pilot", "crm_pilot", then "zoominfo_3", "crm_100" (datasets.py)
F = datasets.files(SET)
STORE, INPUT, ANSWERS, EVIDENCE, REPORT = F.store, F.input, F.answers, F.evidence, F.report
LABEL = F.label  # what the input's industry_types column is ("ZoomInfo" or "CRM"); "" = nothing to compare with

LIMIT    = None  # at most this many accounts this run (e.g. 5 to try it out)
REDO_IDS = []    # account_ids to classify again
WORKERS  = 6     # calls at once (about 60 a minute at the current quota)

## Run
The report is rebuilt every 50 answers and at the end, so close it in Excel first.

In [ ]:
try:
    classify.run(STORE, INPUT, ANSWERS, EVIDENCE, workers=WORKERS, limit=LIMIT, redo_ids=REDO_IDS,
                 on_progress=lambda: report.build(STORE, ANSWERS, REPORT, LABEL, EVIDENCE, INPUT))
finally:
    report.build(STORE, ANSWERS, REPORT, LABEL, EVIDENCE, INPUT)

## The outputs
**Output 1: `REPORT`** (and the same Accounts table as a CSV)

| sheet | what's on it |
|---|---|
| How it decides | the process in plain words, with the thresholds in use |
| Guide | what every column means, what the colours mean, the thresholds |
| Summary | accounts per industry and main business; agreement with the label; how sure; where accounts stopped |
| Review | segments the model itself found unlikely: under 50% when they differ from the ZoomInfo or CRM label, under 30% when they match it. The ones to check. Pick a **Verdict** and add **Notes** (kept when the report is rebuilt) |
| Label differences | where we differ from the label and our answer is sure: skim them, above all the ones we call Other. Verdicts and notes kept too. Accounts that match the label need no check |
| Non-core | companies that do a segment's work alongside a different main business: a count to track, nothing to fix |
| Accounts | every account in the input file: the result, its probability (coloured), main business, match with the label and a one-line rationale; or, if it wasn't classified, the stage it stopped at and why |

**Output 2: `EVIDENCE`**, one JSON line per account with each stage: the crawl, the website text sent (pages, words,
estimated tokens, phrases naming our trades), the website check, the decision (scores, the rule applied, rationale)
and the full input sent to the model.

The report always applies today's thresholds to the saved scores, so changing a threshold in `decide.py` needs only a
report rebuild, not new calls.

## A quick look

In [ ]:
accounts = report.accounts_table(results.load_answers(ANSWERS), LABEL)
print(accounts["AI predicted industry"].value_counts().to_string())
for sheet, part in report.split_for_checking(accounts).items():
    print(f"{sheet}: {len(part)}")
accounts[["Account", f"{LABEL} industry", "AI predicted industry", "Probability", "Rationale"]].head(15)

## Undo a website-issue mark (optional)
If the Accounts sheet says "website issue" (Stopped at: 4 website check) but the site is the company's, list the
account_id here and run the cell: it is classified on the next run and never checked again. If the CRM URL is
fixed instead, Phase 1 crawls the new site.

In [ ]:
UNDO_WEBSITE_ISSUE = []
records = crawl.load_store(STORE)
for account_id in (i.strip().lower() for i in UNDO_WEBSITE_ISSUE):
    if records.get(account_id, {}).get("status") == "wrong_site":
        crawl.mark_site(STORE, records[account_id], "ok", by="manual")
        print(account_id, "cleared")